In [61]:
import importlib
from utils import *
from groq import Groq
from prompts.valid_prompt import valid_reasoning_prompts
from prompts.invalid_prompt import invalid_reasoning_prompts
import re
import time

try:
    import config as app_config
except ModuleNotFoundError:
    import backend.config as app_config

app_config = importlib.reload(app_config)
groq = app_config.groq


In [62]:
#getting the groq client
if not groq:
    raise ValueError("GROQ_API_KEY is missing. Add it to backend/.env.local")

client = Groq(api_key=groq)

In [63]:
#function to generate valid reasoning
def generate_valid_reasoning(problem:str,solution:str):
    chat_completion=client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role":"system",
                "content":valid_reasoning_prompts
            },
            {
                "role":"user",
                "content":f"Problem: {problem}\nSolution: {solution}"
            }
        ]
    )

    return chat_completion.choices[0].message.content

In [64]:
#function to generate invalid reasoning
def generate_invalid_reasoning(problem:str,solution:str):
    chat_completion=client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role":"system",
                "content":invalid_reasoning_prompts
            },
            {
                "role":"user",
                "content":f"Problem: {problem}\nSolution: {solution}"
            }
        ]
    )

    return chat_completion.choices[0].message.content

In [65]:
def extract_reasoning_and_answer(content: str):
    if not content:
        return None, None
        
    parts = re.split(r'(?i)\*{0,2}Answer\*{0,2}\s*:', content)
    
    if len(parts) > 1:
        answer_part = parts[-1].strip()
        
        reasoning_part = "Answer:".join(parts[:-1]).strip()
        
        reasoning_part = re.sub(r'(?i)^\*{0,2}Reasoning\*{0,2}\s*:\s*', '', reasoning_part)
        
        return reasoning_part, answer_part
        
    return None, None

In [66]:
#function to check if the generated answer matches the ground truth answer, with some tolerance for formatting differences

def clean_number(text):
    if not text: 
        return None
    
    numbers = re.findall(r'-?\d+(?:\.\d+)?', str(text))
    
    if numbers:
        return float(numbers[-1]) 
    return None

def is_match(generated_answer, ground_truth):
    gen_num = clean_number(generated_answer)
    truth_num = clean_number(ground_truth)
    
    if gen_num is None or truth_num is None:
        return False
        
    return gen_num == truth_num

In [67]:
#loading the preprocessed data
dataset=load_data("../Data/processed/preprocessed_data.csv")

In [ ]:
#generating the synthetic reasoning data for Critic model pipeline

valid_reasonings=[]
invalid_reasonings=[]
valid_reasoning_results=[]
invalid_reasoning_results=[]
data=dataset[2:5]
for index,row in data.iterrows():
    problem=row["INSTRUCTION"]
    solution=str(row["RESPONSE"]).strip()

    try:
        valid_reasoning=generate_valid_reasoning(problem,solution)
        invalid_reasoning=generate_invalid_reasoning(problem,solution)

        #extracting the reasoning and result from the generated reasoning
        valid_reasoning_steps,valid_reasoning_result=extract_reasoning_and_answer(valid_reasoning)
        invalid_reasoning_steps,invalid_reasoning_result=extract_reasoning_and_answer(invalid_reasoning)

        # print("valid_reasoning_steps:", valid_reasoning_steps)
        # print("valid_reasoning_result:", valid_reasoning_result)
        # print("invalid_reasoning_steps:", invalid_reasoning_steps)
        # print("invalid_reasoning_result:", invalid_reasoning_result)
        #check if the generated answer matches the ground truth answer
        if valid_reasoning_steps and invalid_reasoning_steps:
            if not is_match(valid_reasoning_result, solution):
                print(f"Row {index}: Valid model failed to get the right answer. Skipping.")
                continue

            if not is_match(invalid_reasoning_result, solution):
                print(f"Row {index}: Invalid model failed to arrive at the correct target answer. Skipping.")
                continue

            valid_reasonings.append(valid_reasoning_steps)
            invalid_reasonings.append(invalid_reasoning_steps)

            valid_reasoning_results.append(valid_reasoning_result)
            invalid_reasoning_results.append(invalid_reasoning_result)
        else:
            print(f"Row {index+1}: Formatting error from LLM. Skipping.")

    except Exception as e:
        print(f"API Error on row {index+1}: {e}")
        time.sleep(5)


if len(valid_reasonings) > 0:
    print("Valid Reasoning Steps:", valid_reasonings[0])
    print("Valid Reasoning Result:", valid_reasoning_results[0])
    print("Invalid Reasoning Steps:", invalid_reasonings[0])
    print("Invalid Reasoning Result:", invalid_reasoning_results[0])
else:
    print("No rows passed the validation checks.")


valid_reasoning_steps: None
valid_reasoning_result: None
invalid_reasoning_steps: None
invalid_reasoning_result: None
Row 2: Formatting error from LLM. Skipping.
valid_reasoning_steps: To find out the number of pages Julie should read tomorrow, we first need to determine how many pages she has read so far and how many are left. 

Step 1: Julie read 12 pages yesterday and 24 pages today (twice the number of pages she read yesterday). We find the total number of pages read so far by adding the pages from both days: 12 (yesterday) + 24 (today) = 36 pages.

Step 2: To find the remaining number of pages, we subtract the total pages read so far (36) from the total number of pages in the book (120): 120 - 36 = 84 pages.

Step 3: Now, we need to calculate how many pages Julie should read tomorrow, which is half of the remaining pages. To find half of the remaining pages, we divide the total remaining pages (84) by 2: 84 / 2 = 42 pages.
valid_reasoning_result: 42
invalid_reasoning_steps: Julie 